# AI_EIARI4A_2026: Lab 0 - Baseline Transformer Internals
## 1 Million Parameter Causal Transformer (VUT Prospectus Fine-Tuning)

**Objective:** Build a Decoder-style Transformer with **Causal Masking** to learn facts from the VUT Prospectus 2026.

### 1. Setup and Imports

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import os

print("TensorFlow version:", tf.__version__)

### 2. Building the Causal Transformer Architecture
We use `use_causal_mask=True` in the Attention layer to ensure the model learns to predict the future based ONLY on the past.

In [ ]:
def build_causal_transformer(vocab_size, seq_len=128, embed_dim=128, num_heads=4, ff_dim=512):
    inputs = layers.Input(shape=(seq_len,))
    
    # 1. Embedding + Positional Encoding (Implicit in this simple version)
    x = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)(inputs)
    
    # 2. Causal Transformer Blocks
    for i in range(2):
        # CRITICAL: use_causal_mask=True makes this a Decoder (Generative)
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim, name=f"mha_{i}"
        )(x, x, use_causal_mask=True)
        x = layers.LayerNormalization(epsilon=1e-6)(x + attention_output)
        
        ffn_output = layers.Dense(ff_dim, activation="relu")(x)
        ffn_output = layers.Dense(embed_dim)(ffn_output)
        x = layers.LayerNormalization(epsilon=1e-6)(x + ffn_output)
    
    outputs = layers.Dense(vocab_size, activation="softmax")(x)
    return tf.keras.Model(inputs=inputs, outputs=outputs)

vocab_size = 5000 
model = build_causal_transformer(vocab_size=vocab_size)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")

### 3. Data Pipeline

In [ ]:
with open("vut_prospectus_text.txt", 'r', encoding='utf-8') as f:
    text = f.read()

vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode='int',
    output_sequence_length=129
)

text_chunks = [text[i : i + 500] for i in range(0, len(text) - 500, 100)]

def prepare_dataset(texts, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices(texts)
    vectorize_layer.adapt(ds.batch(64))
    def split_input_target(chunk): return chunk[:, :-1], chunk[:, 1:]
    return ds.batch(batch_size).map(vectorize_layer).map(split_input_target).prefetch(tf.data.AUTOTUNE)

dataset = prepare_dataset(text_chunks)

### 4. Training (The 'Learning' Phase)
Run this for 20-30 epochs. You should see the loss drop below 2.0 for best results.

In [ ]:
model.fit(dataset, epochs=25)

### 5. Interactive Chatbot
**Note**: If the bot outputs random words, it needs more training! Try starting a sentence like *"VUT is located..."*

In [ ]:
def generate_response(prompt, length=40, seq_len=128):
    tokens = vectorize_layer([prompt])
    # Pre-pad to seq_len
    if tokens.shape[1] > seq_len: tokens = tokens[:, -seq_len:]
    else:
        pad = np.zeros((1, seq_len - tokens.shape[1]), dtype=np.int32)
        tokens = tf.convert_to_tensor(np.concatenate([pad, tokens.numpy()], axis=1), dtype=tf.int32)
        
    vocab = vectorize_layer.get_vocabulary()
    result = prompt
    
    for _ in range(length):
        preds = model.predict(tokens, verbose=0)
        next_id = np.argmax(preds[0, -1, :])
        
        tokens_np = np.roll(tokens.numpy(), -1, axis=1)
        tokens_np[0, -1] = next_id
        tokens = tf.convert_to_tensor(tokens_np, dtype=tf.int32)
        
        word = vocab[next_id]
        if word == "": break
        result += " " + word
    return result

chat_history = []
output_area = widgets.Output(layout={'border': '1px solid #ccc', 'height': '400px', 'overflow_y': 'scroll'})
input_box = widgets.Text(placeholder='Start a sentence...', layout={'width': '80%'})
send_btn = widgets.Button(description='Send', button_style='info')

def render():
    with output_area:
        clear_output()
        for speaker, text in chat_history:
            bg = "#f0f0f0" if speaker == "You" else "#e3f2fd"
            display(HTML(f"<div style='margin-bottom:10px; padding:10px; background:{bg}; border-radius:10px;'><b>{speaker}:</b> {text}</div>"))

def on_send(b):
    text = input_box.value
    if not text: return
    input_box.value = ''
    chat_history.append(("You", text))
    render()
    
    res = generate_response(text)
    chat_history.append(("VUT-Bot", res))
    render()

send_btn.on_click(on_send)
display(HTML("<h3 style='color:#002F6E'>VUT Prospectus Chatbot</h3>"))
display(output_area, widgets.HBox([input_box, send_btn]))